# Association Rules and Apriori — Épicerie Laurentienne
**Abdelhafid Masmi — Practice assignment**

In this notebook, we generate synthetic grocery transactions, prepare the baskets, mine association rules, and build a recommendation engine. We follow Parts A–F of the assignment.

We use the supplied generator with seed 404. Running the notebook creates `transactions.csv` and `association_rules.json` in the current folder. The data represents a simulated shop, not real customers.

Required packages: `numpy`, `pandas`, and `mlxtend`. Run the cells in order in a dedicated homework folder; generation replaces `transactions.csv`.

In this section, we import the packages we need. Pandas manages the tables, TransactionEncoder converts baskets into boolean data, and mlxtend provides Apriori and rule generation. We also import JSON and timing tools.

In [1]:
import json
from pathlib import Path
from time import perf_counter
import numpy as np
import pandas as pd
from IPython.display import display
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori, association_rules

## Generate the synthetic transactions
The supplied generator creates themed baskets, random filler items, a common bag and two rare products. The fixed seed makes this exercise reproducible. Dates are randomly assigned across 2026; they do not model real seasonality.

In this section, we generate 3,000 synthetic receipts using the supplied script. We combine shopping themes with random products, a common reusable bag, and two rare products. We use a fixed seed so we can reproduce the data. The dates are random and do not simulate seasonal demand.

In [2]:
import numpy as np
import pandas as pd

rng = np.random.default_rng(404)
N_TRANSACTIONS = 3000

# --- themed baskets: (trigger item, companions, probability each companion joins) ---
THEMES = {
    'bbq':       (['Charcoal'], ['Steak', 'Burger Buns', 'Ketchup', 'Beer'],       [.82, .78, .65, .55]),
    'pasta':     (['Pasta'],    ['Tomato Sauce', 'Parmesan', 'Red Wine', 'Garlic'], [.80, .68, .52, .60]),
    'breakfast': (['Coffee'],   ['Croissants', 'Orange Juice', 'Whole Milk'],       [.62, .55, .58]),
    'baby':      (['Diapers'],  ['Baby Wipes', 'Beer', 'Baby Food'],                [.85, .48, .60]),
    'salad':     (['Lettuce'],  ['Tomatoes', 'Cucumber', 'Olive Oil'],              [.70, .58, .45]),
}
THEME_NAMES = list(THEMES)
THEME_WEIGHTS = [.22, .22, .20, .16, .20]

# --- filler items, drawn at random so every basket is not pure theme ---------
FILLER = ['Bananas', 'Apples', 'Chicken Breast', 'Rice', 'Eggs', 'Butter',
          'Cheddar', 'Yogurt', 'Cereal', 'Bread', 'Onions', 'Carrots',
          'Frozen Pizza', 'Paper Towels', 'Dish Soap', 'Sparkling Water',
          'Potato Chips', 'Chocolate', 'Tea', '2% Milk']
FILLER_P = np.array([6, 5, 5, 4, 6, 4, 4, 4, 3, 7, 5, 4, 3, 3, 3, 4, 4, 4, 3, 5], float)
FILLER_P /= FILLER_P.sum()

RARE = ['Truffle Oil', 'Saffron']          # planted below min_support

# Every themed item can ALSO be bought on its own. Without this the rules come out
# with confidence = 1.000, which never happens in real transaction data and would
# make the exercise trivial.
THEME_ITEMS = sorted({i for trig, comp, _ in THEMES.values() for i in trig + comp})

rows = []
for tid in range(1, N_TRANSACTIONS + 1):
    basket = set()

    # 1. most baskets follow a theme
    if rng.random() < 0.80:
        theme = rng.choice(THEME_NAMES, p=THEME_WEIGHTS)
        triggers, companions, probs = THEMES[theme]
        basket.update(triggers)
        for item, prob in zip(companions, probs):
            if rng.random() < prob:
                basket.add(item)

    # 2. random filler items
    for item in rng.choice(FILLER, size=rng.integers(1, 6), replace=False,
                           p=FILLER_P):
        basket.add(str(item))

    # 3. themed items bought independently -> breaks the perfect rules
    if rng.random() < 0.45:
        for item in rng.choice(THEME_ITEMS, size=rng.integers(1, 4), replace=False):
            basket.add(str(item))

    # 4. THE TRAP: a near-universal item that inflates confidence but not lift
    if rng.random() < 0.84:
        basket.add('Reusable Bag')

    # 5. rare items, far below any sensible min_support
    if rng.random() < 0.004:
        basket.add(str(rng.choice(RARE)))

    for item in sorted(basket):
        rows.append((tid, item))

df = pd.DataFrame(rows, columns=['TransactionID', 'Item'])

# give every transaction a date across one year (for the seasonality question)
dates = pd.to_datetime('2026-01-01') + pd.to_timedelta(
    rng.integers(0, 365, N_TRANSACTIONS), unit='D')
date_map = dict(zip(range(1, N_TRANSACTIONS + 1), dates))
df.insert(1, 'Date', df.TransactionID.map(date_map).dt.strftime('%Y-%m-%d'))

df.to_csv('transactions.csv', index=False)

print(df.head(10).to_string(index=False))
print(f'\n{len(df):,} rows  |  {df.TransactionID.nunique():,} transactions  |  '
      f'{df.Item.nunique()} distinct items')
print(f'average basket size: {len(df) / df.TransactionID.nunique():.2f} items')


 TransactionID       Date           Item
             1 2026-10-09        Bananas
             1 2026-10-09         Coffee
             1 2026-10-09     Croissants
             1 2026-10-09   Reusable Bag
             1 2026-10-09     Whole Milk
             2 2026-02-04    Burger Buns
             2 2026-02-04 Chicken Breast
             2 2026-02-04       Cucumber
             2 2026-02-04        Diapers
             2 2026-02-04        Lettuce

21,578 rows  |  3,000 transactions  |  44 distinct items
average basket size: 7.19 items


## Part A — Understand and prepare the data
### Q1. Transactions, catalogue size and average basket size

In this section, we read the CSV and check the data. We count receipts and products, calculate the average number of distinct products per basket, and check duplicate rows.

In [3]:
df = pd.read_csv("transactions.csv")
assert not df[["TransactionID", "Item"]].isna().any().any()
df['Item'] = df['Item'].str.strip()
basket_series = df.groupby("TransactionID")['Item'].apply(lambda x: sorted(set(x)))
baskets = basket_series.tolist()
basket_sizes = basket_series.apply(len)
print(f"Transactions: {len(baskets):,}")
print(f"Distinct items: {df.Item.nunique()}")
print(f"Average basket size: {basket_sizes.mean():.2f}")
print(f"Duplicate transaction/item rows: {df.duplicated(['TransactionID', 'Item']).sum()}")
display(df.head())

Transactions: 3,000
Distinct items: 44
Average basket size: 7.19
Duplicate transaction/item rows: 0


,TransactionID,Date,Item
0,1,2026-10-09,Bananas
1,1,2026-10-09,Coffee
2,1,2026-10-09,Croissants
3,1,2026-10-09,Reusable Bag
4,1,2026-10-09,Whole Milk


**Results:** There are 3,000 receipts and 44 distinct products. The average customer basket contains 7.19 different items. There are no duplicate transaction/product rows. This agrees with the dataset size in the PDF.

Each row is a scanned product, so row count is not transaction count. The average counts distinct products per basket, not quantities. Product capitalization is preserved so the PDF test baskets match the catalogue.

### Q2. Boolean table

In this section, we convert the baskets into a boolean table. We create one row per receipt and one column per product. True means that a customer bought the product in that receipt.

In [4]:
te = TransactionEncoder()
basket_table = pd.DataFrame(te.fit(baskets).transform(baskets), columns=te.columns_, index=basket_series.index)
print("Shape (transactions × products):", basket_table.shape)
display(basket_table.iloc[:5, :8])

Shape (transactions × products): (3000, 44)


,2% Milk,Apples,Baby Food,Baby Wipes,Bananas,Beer,Bread,Burger Buns
TransactionID,,,,,,,,
1,False,False,False,False,True,False,False,False
2,False,False,False,False,False,False,False,True
3,False,False,False,False,False,False,False,False
4,False,True,False,False,False,False,False,False
5,False,True,False,False,False,True,True,True


**Results:** The table has 3,000 rows and 44 columns. Rows represent receipts, not individual scanned items. Each cell records whether one product is present.

Each True means the product is present in that transaction. Multiple scans of the same product count once; this is the presence table Apriori needs.

### Q3. Most and least frequent items

In this section, we calculate the percentage of baskets containing each product. We display the ten most frequent and five least frequent products to understand popularity before mining rules.

In [5]:
frequency = (basket_table.mean() * 100).sort_values(ascending=False).rename("baskets_percent")
display(frequency.head(10).to_frame())
display(frequency.tail(5).to_frame())
print(f"Reusable Bag appears in {frequency['Reusable Bag']:.2f}% of baskets.")

,baskets_percent
Reusable Bag,85.333333
Bread,23.366667
Pasta,21.866667
Charcoal,21.800000
Beer,20.366667
Eggs,20.200000
Bananas,19.900000
Steak,19.300000
Coffee,19.133333
Burger Buns,18.366667


,baskets_percent
Tea,10.633333
Frozen Pizza,10.566667
Dish Soap,10.166667
Saffron,0.100000
Truffle Oil,0.100000


Reusable Bag appears in 85.33% of baskets.


**Results:** Reusable Bag occurs in 85.33% of receipts. Truffle Oil and Saffron each occur in only three receipts, or 0.10%. This big difference matters: a common product can achieve high confidence without a strong relationship to the starting product.

The percentage uses all baskets as the denominator. Reusable Bag is common; the rare products have very little evidence. Popularity alone does not show a useful product association.

### Q4. Prediction BEFORE mining
We predict that Reusable Bag will dominate the rules ranked by confidence because it occurs in most baskets. Its lift should usually be near 1: knowing another product is present may add little information about whether the customer buys a bag. We will check this in Part C.

### Q5. Two names for one product
The exercise generator explicitly labels Whole Milk and 2% Milk as a split product. In real grocery data these are different fat levels, so We would verify the product IDs before merging. For this synthetic exercise, merging them into Milk would combine their basket presence without double counting baskets containing both. It can change support and which milk rules pass a threshold. We keep all 44 original items for the main analysis.

In this section, we compare the two milk names with their combined basket presence. We count a basket containing both names only once. We keep the original product names for the remaining analysis.

In [6]:
milk_union = basket_table[["Whole Milk", "2% Milk"]].any(axis=1)
display(pd.Series({"Whole Milk": frequency["Whole Milk"], "2% Milk": frequency["2% Milk"], "Merged Milk": milk_union.mean()*100}, name="baskets_percent").to_frame())

,baskets_percent
Whole Milk,12.066667
2% Milk,17.366667
Merged Milk,27.100000


**Results:** Whole Milk appears in 12.07% of baskets and 2% Milk in 17.37%. The merged presence is 27.10%, not their sum, because some baskets contain both. Merging can move Milk and its combinations above a support cutoff.

## Part B — Mine rules
### Q6. Support 0.02 and confidence 0.30
Support 0.02 requires at least 60 out of 3,000 baskets. Confidence 0.30 requires the consequent in at least 30% of baskets containing the antecedent. I use no maximum itemset length, so the experiment is not silently limited to pairs or triples.

In this section, we run Apriori with support 0.02 and generate rules with confidence 0.30. We keep these results as our baseline for the recommendation system.

In [7]:
def mine(table, min_support):
    itemsets = apriori(table, min_support=min_support, use_colnames=True, low_memory=True)
    if itemsets.empty or not itemsets.itemsets.apply(len).ge(2).any():
        rules = pd.DataFrame(columns=["antecedents", "consequents", "support", "confidence", "lift"])
    else:
        rules = association_rules(itemsets, metric="confidence", min_threshold=0.30)
    return itemsets, rules

frequent_itemsets, rules = mine(basket_table, 0.02)
print("Frequent itemsets:", len(frequent_itemsets))
print("Rules:", len(rules))

Frequent itemsets: 1072
Rules: 2357


**Results:** At support 0.02 and confidence 0.30, Apriori returns 1,072 frequent itemsets and 2,357 rules. These are the baseline rules used in Parts C–F. The threshold experiment does not replace this baseline.

### Q7. Support threshold experiment
Confidence stays at 0.30 for every run. Time includes both itemset mining and rule generation and varies by computer. `low_memory=True` reduces candidate memory usage; the lower thresholds may still take time.

In this section, we repeat mining with five support thresholds. We keep confidence at 0.30 and measure itemsets, rules, and total runtime so we can compare the effects fairly.

In [8]:
experiments = []
all_rule_items = set()
for threshold in [0.001, 0.01, 0.02, 0.10, 0.20]:
    start = perf_counter()
    its, rs = mine(basket_table, threshold)
    elapsed = perf_counter() - start
    experiments.append({"min_support": threshold, "frequent_itemsets": len(its), "rules": len(rs), "time_seconds": round(elapsed, 3)})
    for a, c in zip(rs.antecedents, rs.consequents):
        all_rule_items.update(a | c)
    print(f"Finished support={threshold}: {len(its):,} itemsets, {len(rs):,} rules, {elapsed:.3f}s")
threshold_results = pd.DataFrame(experiments)
display(threshold_results)

Finished support=0.001: 76,341 itemsets, 456,827 rules, 10.148s
Finished support=0.01: 3,239 itemsets, 10,851 rules, 0.210s
Finished support=0.02: 1,072 itemsets, 2,357 rules, 0.043s
Finished support=0.1: 100 itemsets, 123 rules, 0.008s
Finished support=0.2: 6 itemsets, 0 rules, 0.002s


,min_support,frequent_itemsets,rules,time_seconds
0,0.001,76341,456827,10.148
1,0.010,3239,10851,0.210
2,0.020,1072,2357,0.043
3,0.100,100,123,0.008
4,0.200,6,0,0.002


**Results:** Support 0.001 produces 76,341 itemsets and 456,827 rules, taking about 10.19 seconds on the recorded run. At 0.02 there are 1,072 itemsets and 2,357 rules. At 0.20 there are only six itemsets and no rules. Low support increases both work and the number of patterns; high support removes all usable associations in this run. Timings will change if we run the notebook again.

### Q8. Too low versus too high

In this section, we compare the lowest and highest support settings. We use the actual counts to explain why very low support creates too many patterns and very high support removes useful rules.

In [9]:
lo, hi = threshold_results.iloc[0], threshold_results.iloc[-1]
print(f"At support {lo.min_support:g}, I get {int(lo.frequent_itemsets):,} itemsets and {int(lo.rules):,} rules.")
print(f"At support {hi.min_support:g}, I get {int(hi.frequent_itemsets):,} itemsets and {int(hi.rules):,} rules.")

At support 0.001, I get 76,341 itemsets and 456,827 rules.
At support 0.2, I get 6 itemsets and 0 rules.


A very low threshold admits combinations supported by only a few receipts, creates many candidates and rules, and can promote chance patterns. A high threshold removes less popular but useful themed combinations; a fast result can still be a poor recommender. The counts above show both effects.

### Q9. Products absent from rules
The wording “whatever you do” is too strong: threshold and confidence changes can allow rare products to appear. We report the actual results over the requested thresholds instead of assuming that exactly two products disappear.

In this section, we identify products absent from every rule in the threshold experiment. We check how many baskets contain the two rare products and distinguish low product frequency from the conditions needed for a rule.

In [10]:
absent = sorted(set(basket_table.columns) - all_rule_items)
print("Absent from rules across the five requested thresholds:", absent)
display(frequency.loc[["Truffle Oil", "Saffron"]].to_frame())
for item in ["Truffle Oil", "Saffron"]:
    print(item, "basket count:", int(basket_table[item].sum()), "appears in tested rules:", item in all_rule_items)

Absent from rules across the five requested thresholds: ['Saffron', 'Truffle Oil']


,baskets_percent
Truffle Oil,0.1
Saffron,0.1


Truffle Oil basket count: 3 appears in tested rules: False
Saffron basket count: 3 appears in tested rules: False


**Results:** Saffron and Truffle Oil are absent from rules at every tested threshold. Each has three baskets. At support 0.001, its single-product itemset can qualify, but a rule also needs a qualifying co-occurring itemset and confidence of at least 0.30. These conditions are not met here. Their absence in this experiment does not prove that no possible threshold could ever produce a rule.

At support 0.02 these rare items cannot qualify because even their individual support is below 2%. At 0.001, an item needs only three baskets, so rare-item rules can become possible. Any such rules need caution because there are few observations. Absence may also result from the confidence cutoff or insufficient co-occurrence.

## Part C — Evaluate rules
### Q10. Plain-language definitions
- **Support:** the share of all baskets containing both sides of a rule.
- **Confidence:** among baskets containing the starting products, the share also containing the suggested products.
- **Lift:** how much more common the suggested products are in those baskets compared with their overall popularity.

### Q11–12. Simple rules ranked by confidence and lift
Both lists use the same baseline rules from support 0.02 and confidence 0.30.

In this section, we keep simple rules with one product on each side. We rank the same rules by confidence and then by lift to compare what each measure emphasizes.

In [11]:
simple_rules = rules[(rules.antecedents.apply(len)==1) & (rules.consequents.apply(len)==1)].copy()
def readable(rs):
    out = rs[["antecedents", "consequents", "support", "confidence", "lift"]].copy()
    for col in ["antecedents", "consequents"]:
        out[col] = out[col].apply(lambda x: ", ".join(sorted(x)))
    return out.reset_index(drop=True)
top_confidence = simple_rules.sort_values(["confidence", "lift"], ascending=False).head(10)
top_lift = simple_rules.sort_values(["lift", "confidence"], ascending=False).head(10)
print("Top 10 by confidence")
display(readable(top_confidence))
print("Top 10 by lift")
display(readable(top_lift))

Top 10 by confidence


,antecedents,consequents,support,confidence,lift
0,Dish Soap,Reusable Bag,0.092333,0.908197,1.064293
1,Sparkling Water,Reusable Bag,0.130000,0.886364,1.038707
2,Lettuce,Reusable Bag,0.158333,0.886194,1.038509
3,Paper Towels,Reusable Bag,0.099333,0.884273,1.036257
4,Rice,Reusable Bag,0.121000,0.872596,1.022574
5,Onions,Reusable Bag,0.143000,0.868421,1.017681
6,Orange Juice,Reusable Bag,0.109667,0.868074,1.017274
7,Tomatoes,Reusable Bag,0.124667,0.865741,1.014540
8,Ketchup,Reusable Bag,0.140667,0.864754,1.013384
9,Apples,Reusable Bag,0.154000,0.863551,1.011974


Top 10 by lift


,antecedents,consequents,support,confidence,lift
0,Baby Wipes,Diapers,0.106000,0.729358,4.465456
1,Diapers,Baby Wipes,0.106000,0.648980,4.465456
2,Baby Food,Diapers,0.078333,0.705706,4.320647
3,Diapers,Baby Food,0.078333,0.479592,4.320647
4,Tomatoes,Lettuce,0.109333,0.759259,4.249585
5,Lettuce,Tomatoes,0.109333,0.611940,4.249585
6,Baby Food,Baby Wipes,0.066667,0.600601,4.132573
7,Baby Wipes,Baby Food,0.066667,0.458716,4.132573
8,Cucumber,Lettuce,0.090333,0.711286,3.981079
9,Lettuce,Cucumber,0.090333,0.505597,3.981079


**Interpretation:** Every top-confidence rule points to Reusable Bag. For example, Dish Soap → Reusable Bag has confidence 90.82% but lift only 1.064. The bag baseline is already 85.33%, so this rule adds little information. In contrast, Baby Wipes → Diapers has confidence 72.94% and lift 4.465. Diapers are much less common overall, so their presence with wipes is a stronger association. The lift list focuses on baby and salad products.

### Q13. Compare rankings and check the prediction

In this section, we check our prediction about Reusable Bag. We count how often it appears in the confidence ranking and compare its rule confidence with its overall frequency.

In [12]:
bag_count = top_confidence.consequents.apply(lambda s: "Reusable Bag" in s).sum()
print(f"{bag_count} of the top 10 confidence rules suggest Reusable Bag.")
print(f"Bag baseline frequency: {frequency['Reusable Bag']/100:.3f}")
bag_rules = simple_rules[simple_rules.consequents.apply(lambda s: "Reusable Bag" in s)]
display(readable(bag_rules.sort_values("confidence", ascending=False).head(5)))
print("The lift list suggests:", sorted(set().union(*top_lift.consequents.tolist())))

10 of the top 10 confidence rules suggest Reusable Bag.
Bag baseline frequency: 0.853


,antecedents,consequents,support,confidence,lift
0,Dish Soap,Reusable Bag,0.092333,0.908197,1.064293
1,Sparkling Water,Reusable Bag,0.130000,0.886364,1.038707
2,Lettuce,Reusable Bag,0.158333,0.886194,1.038509
3,Paper Towels,Reusable Bag,0.099333,0.884273,1.036257
4,Rice,Reusable Bag,0.121000,0.872596,1.022574


The lift list suggests: ['Baby Food', 'Baby Wipes', 'Cucumber', 'Diapers', 'Lettuce', 'Tomatoes']


**Prediction check:** Our Q4 prediction was correct: all ten top-confidence rules suggest the bag. Ranking simple rules by lift removes the popularity advantage and produces a different top-ten list. High confidence alone would give very similar suggestions to many shoppers.

High confidence is easy for Reusable Bag because its baseline frequency is already high. A confidence close to that baseline means the antecedent adds little information, giving lift near 1. Lift divides out this popularity advantage, so themed products such as barbecue and baby products can rank highly despite lower overall frequency. The displayed counts and bag rules test our prediction directly. The lists need not be completely disjoint, but they emphasize different associations.

### Q14. Confidence 0.87, lift 1.02
We would not prioritize this rule for a personalized recommendation. Its confidence is high mainly because the suggested item is already common; lift 1.02 is only a small improvement over the baseline. We would prefer stronger, sufficiently supported associations and verify their value with customer testing.

## Part D — Recommendation engine
### Q15. Implement recommendations
Filter lift > 1 first, match antecedents that are subsets of the basket, exclude owned items, keep the highest-lift suggestion per product, then rank. Multi-item consequents are unpacked into individual suggestions. Equal lifts are resolved by confidence and then product name for reproducible results.

In this section, we implement the recommendation function. We filter lift above 1, match antecedents to the basket, exclude products already selected, and keep the highest-lift rule for each suggestion.

In [13]:
def recommend(basket, rules, top_n=3):
    basket = set(basket)
    if top_n <= 0:
        return []
    eligible = rules[rules['lift'] > 1].copy()
    best = {}
    for _, row in eligible.iterrows():
        if set(row["antecedents"]).issubset(basket):
            for item in set(row["consequents"]) - basket:
                candidate = {"item": item, "lift": float(row['lift']),
                             "confidence": float(row['confidence']), "support": float(row["support"]),
                             "because": sorted(row["antecedents"])}
                if item not in best or (candidate['lift'], candidate['confidence']) > (best[item]['lift'], best[item]['confidence']):
                    best[item] = candidate
    return sorted(best.values(), key=lambda r: (-r['lift'], -r['confidence'], r["item"]))[:top_n]

filtered_rules = rules[rules.lift > 1].copy()

### Q16. Four test baskets

In this section, we test the function with Charcoal, Diapers, Pasta with Garlic, and Coffee. We display the suggested products and the rules that support them.

In [14]:
for basket in [{"Charcoal"}, {"Diapers"}, {"Pasta", "Garlic"}, {"Coffee"}]:
    print("Basket:", sorted(basket))
    display(pd.DataFrame(recommend(basket, filtered_rules)))

Basket: ['Charcoal']


,item,lift,confidence,support,because
0,Burger Buns,4.587156,0.380734,0.083,[Charcoal]
1,Ketchup,4.587156,0.380734,0.083,[Charcoal]
2,Steak,4.587156,0.380734,0.083,[Charcoal]


Basket: ['Diapers']


,item,lift,confidence,support,because
0,Baby Food,5.976676,0.334694,0.054667,[Diapers]
1,Baby Wipes,5.976676,0.334694,0.054667,[Diapers]
2,Reusable Bag,5.976676,0.334694,0.054667,[Diapers]


Basket: ['Garlic', 'Pasta']


,item,lift,confidence,support,because
0,Red Wine,5.455138,0.376404,0.044667,"[Garlic, Pasta]"
1,Reusable Bag,5.455138,0.376404,0.044667,"[Garlic, Pasta]"
2,Tomato Sauce,5.455138,0.376404,0.044667,"[Garlic, Pasta]"


Basket: ['Coffee']


,item,lift,confidence,support,because
0,Reusable Bag,3.876595,0.390244,0.074667,[Coffee]
1,Whole Milk,3.876595,0.390244,0.074667,[Coffee]
2,Croissants,3.761254,0.433798,0.083000,[Coffee]


**Results:**

| Basket | Recommendations |
|---|---|
| Charcoal | Burger Buns, Ketchup, Steak |
| Diapers | Baby Food, Baby Wipes, Reusable Bag |
| Pasta and Garlic | Red Wine, Reusable Bag, Tomato Sauce |
| Coffee | Reusable Bag, Whole Milk, Croissants |

**Important limitation:** The function follows the PDF and uses all matching rules, including rules with several products on the right. It assigns the whole rule's lift to each suggested product. This can let Reusable Bag inherit a strong lift from a bundle containing other associated products. The bundle's lift is not the individual product's lift. This explains why bags can still appear here even though Part C identified the simple-rule confidence trap. For a production system, We would evaluate single-product consequents separately. We retain the requested algorithm and document this limitation.

The returned products reflect associations observed in these synthetic baskets. A basket with Pasta and Garlic can trigger both single-product and multi-product antecedents. An item already in the basket is never returned.

### Q17–18. Edge cases and fallback

In this section, we test a bag-only basket, a rare-product basket, and an empty basket. When no rule matches, we show a clearly labelled list of popular products.

In [15]:
popular_items = frequency.drop(labels=['Reusable Bag']).index.tolist()
def shop_panel(basket, rules, top_n=3):
    results = recommend(basket, rules, top_n)
    if results:
        return {"label": "Customers who bought these also bought", "items": [r["item"] for r in results]}
    fallback = [item for item in popular_items if item not in set(basket)][:max(0, top_n)]
    return {"label": "Popular products (no matching association rule)", "items": fallback}

for basket in [{"Reusable Bag"}, {"Truffle Oil"}, set()]:
    print("Basket:", sorted(basket), "Rules:", recommend(basket, filtered_rules))
    print("Shop display:", shop_panel(basket, filtered_rules))

Basket: ['Reusable Bag'] Rules: []
Shop display: {'label': 'Popular products (no matching association rule)', 'items': ['Bread', 'Pasta', 'Charcoal']}
Basket: ['Truffle Oil'] Rules: []
Shop display: {'label': 'Popular products (no matching association rule)', 'items': ['Bread', 'Pasta', 'Charcoal']}
Basket: [] Rules: []
Shop display: {'label': 'Popular products (no matching association rule)', 'items': ['Bread', 'Pasta', 'Charcoal']}


**Results:** None of these three baskets triggers a baseline rule. The shop displays Bread, Pasta and Charcoal under the label “Popular products (no matching association rule).” This is a popularity fallback, not a personalized association recommendation.

### Q19 — Check the Diapers → Beer example
Beer does not appear among the three returned suggestions for Diapers in this run. The next cell checks the simple association directly. This distinguishes a rule existing in the data from being selected in the top three.

In this section, we check the Diapers → Beer rule directly. We also check whether Beer is selected among the three highest-ranked recommendations, because a rule can exist without appearing in the top three.

In [16]:
diapers_beer = simple_rules[
    (simple_rules["antecedents"] == frozenset({"Diapers"})) &
    (simple_rules["consequents"] == frozenset({"Beer"}))
]
display(readable(diapers_beer))
print("Beer returned among top 3:", any(r["item"] == "Beer" for r in recommend({"Diapers"}, filtered_rules)))

,antecedents,consequents,support,confidence,lift
0,Diapers,Beer,0.07,0.428571,2.104279


Beer returned among top 3: False


Truffle Oil has no baseline rules because it is below 2% support. An empty basket cannot match a nonempty antecedent. Reusable Bag may produce weak positive-lift rules because sampling noise can push lift slightly above 1; We report what actually happens instead of assuming no rules fire. A production system would validate rule strength and consider a stronger lift or evidence threshold. When no rule fires, show a clearly labelled popular-products panel, excluding products already selected.

### Q19. Diapers and Beer
The Diapers → Beer association is the familiar retail analytics example. Here it is deliberately planted by the synthetic generator. It does not show that diapers cause beer purchases: shopping habits or other shared factors can explain co-occurrence, and observational rules cannot establish causation.

## Part E — Save and reload rules
### Q20. Export JSON
Only baseline rules with lift > 1 are exported. Sets become sorted lists so JSON can represent them. Thresholds and synthetic-data provenance are saved with the rules.

In this section, we save the filtered rules to JSON. We convert product sets into lists and include the thresholds and synthetic-data information so the saved rules can be reused.

In [17]:
records = []
for _, row in filtered_rules.iterrows():
    records.append({"antecedents": sorted(row.antecedents), "consequents": sorted(row.consequents),
                    "support": float(row.support), "confidence": float(row.confidence), "lift": float(row.lift)})
payload = {"metadata": {"synthetic": True, "seed": 404, "transactions": len(baskets),
                        "min_support": 0.02, "min_confidence": 0.30, "min_lift_exclusive": 1}, "rules": records}
Path("association_rules.json").write_text(json.dumps(payload, indent=2), encoding="utf-8")
print("Saved rules:", len(records))

Saved rules: 2124


**Results:** There are 2,124 baseline rules with lift greater than 1, saved in `association_rules.json`. A positive lift is the required filter, but it does not by itself guarantee a reliable or profitable recommendation.

Reload in a fresh cell. This lookup uses the JSON file, not the original rules dataframe.

In this section, we reload the JSON file in a fresh cell. We run a Charcoal recommendation and check that the reloaded rules give the same result as the original rules.

In [18]:
loaded_payload = json.loads(Path("association_rules.json").read_text(encoding="utf-8"))
loaded_rules = pd.DataFrame(loaded_payload["rules"])
for col in ["antecedents", "consequents"]:
    loaded_rules[col] = loaded_rules[col].apply(frozenset)
print("Reloaded rules:", len(loaded_rules))
display(pd.DataFrame(recommend({"Charcoal"}, loaded_rules)))
assert recommend({"Charcoal"}, loaded_rules) == recommend({"Charcoal"}, filtered_rules)
print("Reloaded recommendations match the original.")

Reloaded rules: 2124


,item,lift,confidence,support,because
0,Burger Buns,4.587156,0.380734,0.083,[Charcoal]
1,Ketchup,4.587156,0.380734,0.083,[Charcoal]
2,Steak,4.587156,0.380734,0.083,[Charcoal]


Reloaded recommendations match the original.


**Results:** The notebook reloads all 2,124 rules from disk. The assertion confirms that the Charcoal recommendations match the original results. This demonstrates that online lookup can use the saved file without mining again.

### Q21. Offline versus online
Apriori scans historical baskets and generates many candidates, especially at low support. Running it for each visitor wastes work and makes customers wait; concurrent visits could overload the website. Mine in a scheduled offline job, then serve recommendations using saved rules.

### Q22. Refresh and seasonality
We would start with weekly re-mining on a rolling window of recent transactions and monitor changes. Promotions or catalogue changes may justify extra refreshes. For barbecue products, use a summer-specific window or seasonal rule set with enough receipts, and compare it with year-round results. These synthetic dates are random, so this dataset cannot validate a real seasonal effect.

## Part F — Evaluate and reflect
### Q23. Catalogue coverage
Coverage is the number of distinct products in filtered-rule consequents divided by the original catalogue size. For these rules, disjoint antecedents and consequents make every consequent potentially reachable from a matching basket.

In this section, we calculate catalogue coverage. We count the distinct products that appear on the right side of our filtered rules and compare that count with all 44 catalogue products.

In [19]:
recommendable = set().union(*filtered_rules.consequents.tolist()) if len(filtered_rules) else set()
missing = sorted(set(basket_table.columns) - recommendable)
print(f"Coverage: {len(recommendable)}/{len(basket_table.columns)} items ({len(recommendable)/len(basket_table.columns):.1%})")
print("Products never recommended by association rules:", missing)

Coverage: 22/44 items (50.0%)
Products never recommended by association rules: ['2% Milk', 'Apples', 'Bananas', 'Bread', 'Butter', 'Carrots', 'Cereal', 'Cheddar', 'Chicken Breast', 'Chocolate', 'Dish Soap', 'Eggs', 'Frozen Pizza', 'Onions', 'Paper Towels', 'Potato Chips', 'Rice', 'Saffron', 'Sparkling Water', 'Tea', 'Truffle Oil', 'Yogurt']


**Results:** The system can recommend 22 of the 44 products, giving 50% catalogue coverage. Rare products and many independent filler products are not in the filtered consequents. The fallback can display other products, but it does not increase association-rule coverage.

Coverage below 100% means some catalogue products cannot receive an association-based suggestion. It is a problem if the shop needs to promote rare or new items. Add popular-product or category/content-based fallbacks, collect more data, and consider lower support only with evidence checks. This is catalogue coverage, not the percentage of customer baskets that receive suggestions.

### Q24. Two limitations and possible additions
1. **New or rare products:** Apriori needs enough co-purchases. Add recommendations based on product attributes and categories for cold-start products.
2. **Personal preferences and purchase order:** These basket rules do not model a customer history or which purchase came first. With appropriate consent, add customer-history personalization or sequential models, then evaluate on held-out future data.

### Honest notes
This is synthetic practice data with deliberately planted patterns, not proof of real customer behavior. The supplied generator differs from some wording in the PDF: it uses an 84% bag probability, the rare items can enter rules at sufficiently low thresholds, and milk names represent different real products despite being declared aliases in this exercise. The notebook reports computed evidence rather than forcing expected answers. Runtime numbers depend on the computer. `lift > 1` meets the assignment but does not alone establish statistical reliability or business value.